In [1]:
import sys
sys.path.append('../')
import pandas as pd
import karman
import torch
from omegaconf import OmegaConf
from tft_torch import tft
import tft_torch.loss as tft_loss
from torch import nn
import torch.nn.init as init
import numpy as np

device=torch.device('cpu')

karman_dataset=karman.KarmanDataset(thermo_path='../data/satellites_data_w_sw_2mln.csv',
                            min_date=pd.to_datetime("2000-07-29 00:59:47"),
                            max_date=pd.to_datetime("2024-05-31 23:59:32"),
                            normalization_dict=None,
                            omni_indices_path='../data/omniweb_data/merged_omni_indices.csv',
                            nrlmsise00_path='../data/nrlmsise00_data/nrlmsise00_time_series.csv',
                            #omni_path='omniweb_1min_data_2001_2022.h5',
                           )

Loading Omni indices.


/home/ga00693/2024-HL-ThermCL/notebooks/../karman/dataset.py:453: FutureWarning: DataFrame.interpolate with method=pad is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  self.time_series_data[data_name]["data"] = self.time_series_data[data_name]["data"].interpolate(method="pad")
/home/ga00693/2024-HL-ThermCL/notebooks/../karman/dataset.py:456: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  self.time_series_data[data_name]["data"] = (self.time_series_data[data_name]["data"].resample(f"{resolution}T").ffill())


Creating thermospheric density dataset
Removing from the data errors in mean absolute percentage error 200% or more in the density (between nrlmsise00 and ground truth)
loading it from file
Used features: Index(['tudelft_thermo__altitude__[m]', 'tudelft_thermo__latitude__[deg]',
       'celestrack__ap_average__',
       'space_environment_technologies__f107_obs__',
       'space_environment_technologies__f107_average__',
       'space_environment_technologies__s107_obs__',
       'space_environment_technologies__s107_average__',
       'space_environment_technologies__m107_obs__',
       'space_environment_technologies__m107_average__',
       'space_environment_technologies__y107_obs__',
       'space_environment_technologies__y107_average__', 'JB08__d_st_dt__[K]',
       'tudelft_thermo__longitude__[deg]_sin',
       'tudelft_thermo__longitude__[deg]_cos', 'all__day_of_year__[d]_sin',
       'all__day_of_year__[d]_cos', 'all__seconds_in_day__[s]_sin',
       'all__seconds_in_day__[s]

/home/ga00693/2024-HL-ThermCL/notebooks/../karman/dataset.py:480: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  self.time_series_data[data_name]["data"] = (self.time_series_data[data_name]["data"].resample(f"{resolution}T").ffill())



Finished Creating dataset.


In [3]:
def weight_init(m):
    """
    Usage:
        model = Model()
        model.apply(weight_init)
    """
    if isinstance(m, nn.Conv1d):
        init.normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.Conv2d):
        init.xavier_normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.Conv3d):
        init.xavier_normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.ConvTranspose1d):
        init.normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.ConvTranspose2d):
        init.xavier_normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.ConvTranspose3d):
        init.xavier_normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.BatchNorm1d):
        init.normal_(m.weight.data, mean=1, std=0.02)
        init.constant_(m.bias.data, 0)
    elif isinstance(m, nn.BatchNorm2d):
        init.normal_(m.weight.data, mean=1, std=0.02)
        init.constant_(m.bias.data, 0)
    elif isinstance(m, nn.BatchNorm3d):
        init.normal_(m.weight.data, mean=1, std=0.02)
        init.constant_(m.bias.data, 0)
    elif isinstance(m, nn.Linear):
        init.xavier_normal_(m.weight.data)
        if m.bias is not None:
            init.normal_(m.bias.data)
    elif isinstance(m, nn.LSTM):
        for param in m.parameters():
            if len(param.shape) >= 2:
                init.orthogonal_(param.data)
            else:
                init.normal_(param.data)
    elif isinstance(m, nn.LSTMCell):
        for param in m.parameters():
            if len(param.shape) >= 2:
                init.orthogonal_(param.data)
            else:
                init.normal_(param.data)
    elif isinstance(m, nn.GRU):
        for param in m.parameters():
            if len(param.shape) >= 2:
                init.orthogonal_(param.data)
            else:
                init.normal_(param.data)
        for names in m._all_weights:
            for name in filter(lambda n: "bias" in n, names):
                bias = getattr(m, name)
                n = bias.size(0)
                bias.data[: n // 3].fill_(-1.0)
    elif isinstance(m, nn.GRUCell):
        for param in m.parameters():
            if len(param.shape) >= 2:
                init.orthogonal_(param.data)
            else:
                init.normal_(param.data)

In [4]:
# set configuration
data_props = {'num_historical_numeric': karman_dataset[0]['omni_indices'].shape[1]+karman_dataset[0]['msise'].shape[1],
            'num_static_numeric': karman_dataset[0]['instantaneous_features'].shape[0],
            'num_future_numeric': 1,
            }

configuration = {
'model':
    {
        'dropout': 0.05,
        'state_size': 64,
        'output_quantiles': [0.5],
        'lstm_layers': 2,
        'attention_heads': 4
    },
'task_type': 'regression',
'target_window_start': None,
'data_props': data_props,
}

In [5]:
from torch import optim
from torch.utils.data import SequentialSampler, RandomSampler
# initialize tft_model
tft_model = tft.TemporalFusionTransformer(OmegaConf.create(configuration))
# weight init
tft_model.apply(weight_init)
tft_model.to(device)

lr=1e-4
#optimizer:
optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, list(tft_model.parameters())),
    lr=lr,
    amsgrad=True,
)

# create batch as an example
historical_steps = karman_dataset[0]['omni_indices'].shape[0]-1
future_steps = 1


# split the dataset into train, validation, and test
idx_test_fold = 2
test_month_idx = 2 * (idx_test_fold - 1)
validation_month_idx = test_month_idx + 2
print(test_month_idx, validation_month_idx)
karman_dataset._set_indices(
    test_month_idx=[test_month_idx],
    validation_month_idx=[validation_month_idx],
    custom={2024: {"validation": 3, "test": 4}},
)
train_dataset = karman_dataset.train_dataset()
validation_dataset = karman_dataset.validation_dataset()
test_dataset = karman_dataset.test_dataset()
# print(f"Training dataset example: {train_dataset[0].items()}")

train_sampler = RandomSampler(train_dataset, num_samples=len(train_dataset))
validation_sampler = RandomSampler(
    validation_dataset, num_samples=len(validation_dataset)
)
test_sampler = SequentialSampler(test_dataset)

2 4
Creating training, validation and test sets.


25 years to iterate through.: 100%|██████████| 25/25 [00:02<00:00, 11.06it/s]

Train size: 1687843
Validation size: 172138
Test size: 176629


In [7]:
batch_size = 64
num_workers = 0
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=batch_size,
    pin_memory=False,
    num_workers=num_workers,
    sampler=train_sampler,
    drop_last=True,
)
validation_loader = torch.utils.data.DataLoader(
    validation_dataset,
    batch_size=batch_size,
    pin_memory=False,
    num_workers=num_workers,
    sampler=validation_sampler,
    drop_last=False,
)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=batch_size,
    pin_memory=False,
    num_workers=num_workers,
    sampler=test_sampler,
    drop_last=False,
)

In [8]:
epochs = 2
quantiles_tensor = torch.tensor(configuration["model"]["output_quantiles"]).to(device)


In [20]:
def mean_absolute_percentage_error(y_pred,y_true):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def mse(y_pred,y_true):
    return np.mean((y_true - y_pred) ** 2)

losses_per_minibatch={  'q_loss_train':[],'q_risk_train':[],
                        'q_loss_valid':[],'q_risk_valid':[],
                        'nn_mse_train':[],'nrlmsise00_mse_train':[],'nn_mape_train':[],'nrlmsise00_mape_train':[],
                        'nn_mse_valid':[],'nrlmsise00_mse_valid':[],'nn_mape_valid':[],'nrlmsise00_mape_valid':[]}
losses_total={  'q_loss_train':[],'q_risk_train':[],
                'q_loss_valid':[],'q_risk_valid':[],
                'nn_mse_train':[],'nrlmsise00_mse_train':[],'nn_mape_train':[],'nrlmsise00_mape_train':[],
                'nn_mse_valid':[],'nrlmsise00_mse_valid':[],'nn_mape_valid':[],'nrlmsise00_mape_valid':[]}


best_loss_total = np.inf
best_loss = np.inf
best_q_loss = np.inf 
criterion=nn.MSELoss()
for epoch in range(epochs):
    #first training loop:
    q_loss_total=0.
    q_risk_total=0.
    loss_total_nn=0.
    loss_total_nrlmsise00=0.
    mape_total_nn=0.
    mape_total_nrlmsise00=0.
    #we set the model in training mode:
    tft_model.train()
    for batch_idx,el in enumerate(train_loader):
        minibatch = {
                'static_feats_numeric': el['instantaneous_features'].to(device),
                'historical_ts_numeric': torch.cat([el['omni_indices'][:,:-1,:],el['msise'][:,:-1,:]],dim=2).to(device),
                'future_ts_numeric':  el['omni_indices'][:,-1,:].unsqueeze(1).to(device),
                'target': el['target'].to(device)
                }
        #let's store the normalized and unnormalized target density:
        target=el['target'].to(device)
        rho_target=el['ground_truth'].detach().cpu().numpy()
        batch_out=tft_model(minibatch)
        #now the quantiles:
        predicted_quantiles = batch_out['predicted_quantiles']#it's of shape batch_size x future_steps x num_quantiles
        target_nn_median=predicted_quantiles[:, :, 0].squeeze()
        q_loss, q_risk, _ = tft_loss.get_quantiles_loss_and_q_risk(outputs=predicted_quantiles,
                                                                    targets=target,
                                                                    desired_quantiles=quantiles_tensor)
        #now the normalized and unnormalized NN-predicted density:
        #if opt.train_type=='log_exp_residual':
        #    out_nn=torch.tanh(tft_model(minibatch).squeeze())
        #    target_nn=karman_dataset.scale_density(el['exponential_atmosphere'].to(device))+out_nn
        #else:
        #    target_nn=tft_model(minibatch).squeeze()
        rho_nn=karman_dataset.unscale_density(target_nn_median.detach().cpu()).numpy()
        #finally the NRLMSISE-00 ones:
        rho_nrlmsise00=el['nrlmsise00'].detach().cpu().numpy()
        target_nrlmsise00=karman_dataset.scale_density(el['nrlmsise00'].to(device))
        #now the loss computation:
        loss_nn = criterion(target_nn_median, target)
        loss_nrlmsise00 = mse(target_nrlmsise00.detach().cpu().numpy(), target.detach().cpu().numpy())

        # Zeroes the gradient 
        optimizer.zero_grad()

        # Backward pass: compute gradient of the loss with respect to model parameters
        #q_loss.backward()
        loss_nn.backward()

        # Calling the step function on an Optimizer makes an update to its parameters
        optimizer.step()

        #We compute the logged quantities
        #log to wandb:
        #wandb.log({'nn_mse_train':loss_nn.item(),'nrlmsise00_mse_train':loss_nrlmsise00,
        #            'nn_mape_train':mean_absolute_percentage_error(rho_nn, rho_target),
        #            'nrlmsise00_mape_train':mean_absolute_percentage_error(rho_nrlmsise00, rho_target)})
        
        losses_per_minibatch['q_loss_train'].append(q_loss.item())
        losses_per_minibatch['q_risk_train'].append(q_risk.detach().cpu().numpy())
        losses_per_minibatch['nn_mse_train'].append(loss_nn.item())
        losses_per_minibatch['nn_mse_train'].append(loss_nn.item())
        losses_per_minibatch['nrlmsise00_mse_train'].append(loss_nrlmsise00)
        losses_per_minibatch['nn_mape_train'].append(mean_absolute_percentage_error(rho_nn, rho_target))
        losses_per_minibatch['nrlmsise00_mape_train'].append(mean_absolute_percentage_error(rho_nrlmsise00, rho_target))
        #now let's also accumulate them for the overall loss computation in each epoch:
        q_loss_total+=losses_per_minibatch['q_loss_train'][-1]
        q_risk_total+=losses_per_minibatch['q_risk_train'][-1]
        loss_total_nn+=losses_per_minibatch['nn_mse_train'][-1]
        loss_total_nrlmsise00+=losses_per_minibatch['nrlmsise00_mse_train'][-1]
        mape_total_nn+=losses_per_minibatch['nn_mape_train'][-1]
        mape_total_nrlmsise00+=losses_per_minibatch['nrlmsise00_mape_train'][-1]
        
        #Save the best model (this is wrong and should be done on the dataset):
        if loss_nn.item()<best_loss:    
            best_loss=loss_nn.item()

        #Print every 10 minibatches:
        if batch_idx%10:    
            print(f'minibatch: {batch_idx}/{len(train_loader)}, best minibatch loss till now: {best_loss:.4e}, NN MSE: {losses_per_minibatch['nn_mse_train'][-1]:.10f}, nrlmsise00 MSE: {losses_per_minibatch['nrlmsise00_mse_train'][-1]:.10f}, NN MAPE: {losses_per_minibatch['nn_mape_train'][-1]:.3f}, nrlmsise00 MAPE: {losses_per_minibatch['nrlmsise00_mape_train'][-1]:.3f}', end='\r')
    #log to wandb:
    #wandb.log({'nn_mse_train_total':loss_total_nn/len(train_loader),'nrlmsise00_mse_train_total':loss_total_nrlmsise00/len(train_loader),
    #                'nn_mape_train_total':mape_total_nn/len(train_loader),
    #                'nrlmsise00_mape_train_total':mape_total_nrlmsise00/len(train_loader)})
    # over the whole dataset, we take the average of the minibatch losses:
    losses_total['q_loss_train'].append(q_loss_total/len(train_loader))
    losses_total['q_risk_train'].append(q_risk_total/len(train_loader))
    losses_total['nn_mse_train'].append(loss_total_nn/len(train_loader))
    losses_total['nrlmsise00_mse_train'].append(loss_total_nrlmsise00/len(train_loader))
    losses_total['nn_mape_train'].append(mape_total_nn/len(train_loader))
    losses_total['nrlmsise00_mape_train'].append(mape_total_nrlmsise00/len(train_loader))

    #Print at the end of the epoch
    #curr_lr = scheduler.optimizer.param_groups[0]['lr']
    print(" "*300, end="\r")    
    print("\nTraining")
    print(f'Epoch {epoch + 1}/{epochs}, NN MSE (total): {losses_total['nn_mse_train'][-1]:.7f}, nrlmsise00 MSE (total): {losses_total['nrlmsise00_mse_train'][-1]:.7f}, NN MAPE (total): {losses_total['nn_mape_train'][-1]:.3f}, nrlmsise00 MAPE (total): {losses_total['nrlmsise00_mape_train'][-1]:.3f}')
    # Perform a step in LR scheduler to update LR
    #scheduler.step()
    
    #Validation loop:
    loss_total_nn=0.
    loss_total_nrlmsise00=0.
    mape_total_nn=0.
    mape_total_nrlmsise00=0.
    #let's switch the model to evaluation mode:
    tft_model.eval()
    with torch.no_grad():
        for batch_idx,el in enumerate(validation_loader):
            minibatch = {
                    'static_feats_numeric': el['instantaneous_features'].to(device),
                    'historical_ts_numeric': torch.cat([el['omni_indices'][:,:-1,:],el['msise'][:,:-1,:]],dim=2).to(device),
                    'future_ts_numeric':  torch.cat(el['omni_indices'][:,-1,:],el['msise'][:,-1,:],dim=2).unsqueeze(1).to(device),#batch size x future steps x num features
                    'target': el['target'].to(device)
                    }
            #let's store the normalized and unnormalized target density:
            target=el['target'].to(device)
            rho_target=el['ground_truth'].detach().cpu().numpy()
            batch_out=tft_model(minibatch)
            target_nn_median=predicted_quantiles[:, :, 1].squeeze()
            #now the quantiles:
            predicted_quantiles = batch_out['predicted_quantiles']
            q_loss, q_risk, _ = tft_loss.get_quantiles_loss_and_q_risk(outputs=predicted_quantiles,
                                                                        targets=target,
                                                                        desired_quantiles=quantiles_tensor)
            #now the normalized and unnormalized NN-predicted density:
            #if opt.valid_type=='log_exp_residual':
            #    out_nn=torch.tanh(tft_model(minibatch).squeeze())
            #    target_nn=karman_dataset.scale_density(el['exponential_atmosphere'].to(device))+out_nn
            #else:
            #    target_nn=tft_model(minibatch).squeeze()
            rho_nn=karman_dataset.unscale_density(target_nn_median.detach().cpu()).numpy()
            #finally the NRLMSISE-00 ones:
            rho_nrlmsise00=el['nrlmsise00'].detach().cpu().numpy()
            target_nrlmsise00=karman_dataset.scale_density(el['nrlmsise00'].to(device))
            #now the loss computation:
            loss_nn = criterion(target_nn_median, target)
            loss_nrlmsise00 = mse(target_nrlmsise00.detach().cpu().numpy(), target.detach().cpu().numpy())
            #We compute the logged quantities
            #log to wandb:
            #wandb.log({'nn_mse_valid':loss_nn.item(),'nrlmsise00_mse_valid':loss_nrlmsise00,
            #            'nn_mape_valid':mean_absolute_percentage_error(rho_nn, rho_target),
            #            'nrlmsise00_mape_valid':mean_absolute_percentage_error(rho_nrlmsise00, rho_target)})
            
            losses_per_minibatch['q_loss_valid'].append(q_loss.item())
            losses_per_minibatch['q_risk_valid'].append(q_risk.detach().cpu().numpy())
            losses_per_minibatch['nn_mse_valid'].append(loss_nn.item())
            losses_per_minibatch['nn_mse_valid'].append(loss_nn.item())
            losses_per_minibatch['nrlmsise00_mse_valid'].append(loss_nrlmsise00)
            losses_per_minibatch['nn_mape_valid'].append(mean_absolute_percentage_error(rho_nn, rho_target))
            losses_per_minibatch['nrlmsise00_mape_valid'].append(mean_absolute_percentage_error(rho_nrlmsise00, rho_target))
            #now let's also accumulate them for the overall loss computation in each epoch:
            q_loss_total+=losses_per_minibatch['q_loss_valid'][-1]
            q_risk_total+=losses_per_minibatch['q_risk_valid'][-1]
            loss_total_nn+=losses_per_minibatch['nn_mse_valid'][-1]
            loss_total_nrlmsise00+=losses_per_minibatch['nrlmsise00_mse_valid'][-1]
            mape_total_nn+=losses_per_minibatch['nn_mape_valid'][-1]
            mape_total_nrlmsise00+=losses_per_minibatch['nrlmsise00_mape_valid'][-1]
            
            #Save the best model (this is wrong and should be done on the dataset):
            if loss_nn.item()<best_loss:    
                best_loss=loss_nn.item()

            #Print every 10 minibatches:
            if batch_idx%10:    
                print(f'minibatch: {batch_idx}/{len(validation_loader)}, best minibatch loss till now: {best_loss:.4e}, NN MSE: {losses_per_minibatch['nn_mse_valid'][-1]:.10f}, nrlmsise00 MSE: {losses_per_minibatch['nrlmsise00_mse_valid'][-1]:.10f}, NN MAPE: {losses_per_minibatch['nn_mape_valid'][-1]:.3f}, nrlmsise00 MAPE: {losses_per_minibatch['nrlmsise00_mape_valid'][-1]:.3f}', end='\r')
        #log to wandb:
        #wandb.log({'nn_mse_valid_total':loss_total_nn/len(validation_loader),'nrlmsise00_mse_valid_total':loss_total_nrlmsise00/len(validation_loader),
        #                'nn_mape_valid_total':mape_total_nn/len(validation_loader),
        #                'nrlmsise00_mape_valid_total':mape_total_nrlmsise00/len(validation_loader)})
        # over the whole dataset, we take the average of the minibatch losses:
        losses_total['q_loss_valid'].append(q_loss_total/len(validation_loader))
        losses_total['q_risk_valid'].append(q_risk_total/len(validation_loader))
        losses_total['nn_mse_valid'].append(loss_total_nn/len(validation_loader))
        losses_total['nrlmsise00_mse_valid'].append(loss_total_nrlmsise00/len(validation_loader))
        losses_total['nn_mape_valid'].append(mape_total_nn/len(validation_loader))
        losses_total['nrlmsise00_mape_valid'].append(mape_total_nrlmsise00/len(validation_loader))

    print("\nValidation")
    print(f'Epoch {epoch + 1}/{epochs}, NN MSE (total): {losses_total['nn_mse_valid'][-1]:.7f}, nrlmsise00 MSE (total): {losses_total['nrlmsise00_mse_valid'][-1]:.7f}, NN MAPE (total): {losses_total['nn_mape_valid'][-1]:.3f}, nrlmsise00 MAPE (total): {losses_total['nrlmsise00_mape_valid'][-1]:.3f}')
    #updating torch best model:
    if losses_total['nn_mse_valid'][-1] < best_loss_total:
        #log to wandb:
        #wandb.log({'best_nn_mse_valid':losses_total['nn_mse_valid'][-1]})
        #create directory if it does not exist:
        import os
        if not os.path.exists('../models'):
            os.makedirs('../models')
        torch.save(tft_model.state_dict(), f'../models/tft_model_{train_type}_valid_loss_{losses_total['nn_mse_valid'][-1]}_params_{num_params}.torch')
        best_loss_total=losses_total['nn_mse_valid'][-1]


KeyboardInterrupt: 

In [13]:
torch.cat([el['omni_indices'][:,-1,:],el['msise'][:,-1,:]],dim=1)

tensor([[-0.9606,  0.6584, -0.9456,  ...,  0.2142,  0.1851,  0.1536],
        [ 0.7877, -1.0078,  0.0959,  ...,  0.1541,  0.2607,  0.2714],
        [-0.1894,  0.2339, -0.0050,  ...,  0.2633,  0.1687,  0.2685],
        ...,
        [-0.9606,  0.6584, -0.9456,  ...,  0.0873,  0.2331,  0.1556],
        [-0.2259,  0.2058, -0.0885,  ...,  0.2341,  0.0967,  0.0957],
        [ 0.1059, -0.2503, -0.0308,  ...,  0.1726,  0.2850,  0.2391]])

In [70]:
losses_per_minibatch['nn_mape_train'], losses_per_minibatch['nrlmsise00_mape_train']

([96.60118818283081,
  91.05104804039001,
  198.7972855567932,
  122.39665985107422,
  380.20052909851074,
  782.3528289794922,
  1462.3779296875,
  3289.788818359375,
  7979.4769287109375,
  6132.159805297852,
  8306.485748291016,
  6625.691223144531,
  7316.552734375,
  5907.417297363281,
  1538.41552734375,
  1988.7359619140625,
  1235.8972549438477,
  539.2333507537842,
  579.6662330627441,
  489.459228515625,
  507.19285011291504,
  254.60054874420166,
  249.36118125915527,
  179.2807698249817,
  168.31130981445312,
  477.0880699157715,
  305.31110763549805,
  1347.5639343261719,
  430.4600715637207,
  338.66114616394043,
  346.6207504272461,
  429.9197196960449,
  580.0028800964355,
  1133.894157409668,
  517.2860145568848,
  680.6781768798828,
  1580.898094177246,
  502.5496482849121,
  694.550085067749,
  964.6479606628418,
  750.2597808837891,
  1390.958595275879,
  491.21923446655273,
  664.7647857666016,
  1076.4016151428223,
  336.2133979797363,
  386.3198518753052,
  509.0

In [64]:
q_risk.detach().cpu().numpy()

array([ 32.651436, 237.76636 , 142.29762 ], dtype=float32)

In [61]:
batch_out

{'predicted_quantiles': tensor([[[-1.1762, -1.4367, -0.9911]],
 
         [[-1.3358, -1.4567, -0.6662]],
 
         [[-0.4922, -1.5157, -0.3739]],
 
         [[-1.0849, -1.5607, -0.6830]],
 
         [[-0.6876, -1.7412, -0.7191]],
 
         [[-1.1045, -1.5967, -0.5293]],
 
         [[-1.1598, -1.6305, -0.8010]],
 
         [[-1.4016, -1.4018, -0.9678]],
 
         [[-1.0447, -2.1853, -0.7017]],
 
         [[-1.2713, -1.5318, -0.8856]],
 
         [[-1.3617, -1.2347, -0.6737]],
 
         [[-1.0272, -1.5571, -0.3858]],
 
         [[-0.9539, -1.5470, -0.6855]],
 
         [[-1.3114, -1.7632, -0.3086]],
 
         [[-0.9554, -1.3982, -0.3560]],
 
         [[-1.1233, -1.3665, -0.7140]],
 
         [[-0.8377, -2.0275, -0.4153]],
 
         [[-1.7785, -1.0739, -0.8001]],
 
         [[-1.3117, -0.9991, -1.1899]],
 
         [[-0.4649, -2.1588, -0.5580]],
 
         [[-1.0680, -1.7119, -0.9275]],
 
         [[-0.7235, -1.0765, -0.1723]],
 
         [[-0.9456, -1.5475, -0.6309]],
 
         [[

In [62]:
import copy
v1=next(iter(train_loader))
v2=copy.deepcopy(v1)



In [59]:
v2['instantaneous_features'].shape

torch.Size([126, 18])

In [64]:
tft_model(batch)['predicted_quantiles'].shape

torch.Size([126, 1, 3])

In [67]:
v['msise']

tensor([5.2228e-14, 4.4706e-13, 1.0553e-12, 1.0458e-13, 1.6674e-12, 6.6457e-14,
        1.0239e-12, 3.8941e-12, 4.6224e-14, 1.6378e-13, 1.7027e-13, 1.0334e-12,
        7.5006e-13, 1.0828e-13, 8.2281e-14, 2.3160e-13, 6.9722e-14, 3.1020e-11,
        1.3306e-13, 3.0781e-12, 5.4670e-12, 1.6340e-11, 2.0796e-12, 2.4213e-13,
        5.1570e-13, 3.4227e-11, 1.3503e-12, 9.4645e-14, 2.2302e-12, 1.5756e-13,
        1.2207e-12, 1.9148e-12, 1.4683e-13, 1.9136e-13, 1.6968e-13, 2.5812e-13,
        2.9117e-13, 1.8171e-13, 3.1093e-13, 1.0350e-12, 2.4358e-13, 3.1256e-13,
        4.6024e-13, 7.7731e-11, 5.0822e-13, 3.4147e-11, 1.7511e-12, 2.5108e-12,
        1.4585e-13, 8.1650e-12, 1.1461e-13, 5.2493e-14, 1.7868e-12, 1.1816e-13,
        1.3187e-13, 1.5842e-12, 8.6363e-13, 3.9461e-14, 1.0294e-12, 2.2928e-12,
        2.8582e-13, 1.4750e-13, 3.5197e-13, 1.0371e-13, 8.1006e-14, 1.8058e-12,
        7.7697e-13, 2.9684e-12, 1.5841e-12, 3.4472e-12, 2.2338e-12, 5.0081e-13,
        1.0486e-13, 4.2822e-13, 1.9806e-

In [18]:

batch = {
    'static_feats_numeric': karman_dataset[0]['instantaneous_features'].unsqueeze(0),#torch.rand(batch_size, data_props['num_static_numeric'],
                            #           dtype=torch.float32),
    'historical_ts_numeric': karman_dataset[0]['omni_indices'].unsqueeze(0),#torch.rand(batch_size, historical_steps, data_props['num_historical_numeric'],
                             #           dtype=torch.float32),
    'future_ts_numeric':  torch.stack([karman_dataset[0]['instantaneous_features']]*1).unsqueeze(0),#torch.rand(batch_size, future_steps, data_props['num_future_numeric'],
                         #           dtype=torch.float32),
    'target': karman_dataset[0]['target'].unsqueeze(0)
}
# apply model
batch_outputs = tft_model(batch)



IndexError: index 6 is out of bounds for dimension 0 with size 6

In [16]:
batch_outputs.keys()

dict_keys(['predicted_quantiles', 'static_weights', 'historical_selection_weights', 'future_selection_weights', 'attention_scores'])

In [14]:
batch_outputs.keys()

dict_keys(['predicted_quantiles', 'static_weights', 'historical_selection_weights', 'future_selection_weights', 'attention_scores'])

In [10]:
#count total number of parameters
total_params = sum(p.numel() for p in model.parameters())
print(total_params)

863723


torch.Size([49, 10])

In [1]:
 karman_dataset[0]['omni_indices']

NameError: name 'karman_dataset' is not defined